# 3. Data Preparation

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

## 3.1 Load Dataset

In [20]:
df = pd.read_csv('../Datasets/hypertension_dataset.csv')
print(f'Total Data Awal: {df.shape}')

Total Data Awal: (1985, 11)


## 3.2 Handling Missing Values

In [21]:
missing = df.isnull().sum()
print('Missing Values Awal:\n', missing)

# Mengisi missing value pada Medication
df['Medication'] = df['Medication'].astype('object')
df['Medication'] = df['Medication'].fillna('None')

print('\nMissing values pada Medication setelah diisi:', df['Medication'].isnull().sum())
print('\nDistribusi nilai Medication:')
print(df['Medication'].value_counts())

Missing Values Awal:
 Age                   0
Salt_Intake           0
Stress_Score          0
BP_History            0
Sleep_Duration        0
BMI                   0
Medication          799
Family_History        0
Exercise_Level        0
Smoking_Status        0
Has_Hypertension      0
dtype: int64

Missing values pada Medication setelah diisi: 0

Distribusi nilai Medication:
Medication
None             799
Beta Blocker     412
ACE Inhibitor    323
Diuretic         245
Other            206
Name: count, dtype: int64


## 3.3 Encoding Fitur Kategorikal
Karena LightGBM dapat menangani data kategorikal secara native jika diubah menjadi integer tipe kategori, kita akan menggunakan `LabelEncoder` untuk seluruh kolom objek (termasuk target).

In [22]:
cat_cols = df.select_dtypes(include=['object', 'str']).columns
le_dict = {}

df_prepared = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    encoded_values = le.fit_transform(df_prepared[col].astype(str))
    df_prepared[col] = pd.Series(encoded_values, index=df_prepared.index).astype('category')
    le_dict[col] = le

print("=== TABEL 1: Data Hasil Encoding ===")
display(df_prepared.head())

mapping_rows = []
for col, le in le_dict.items():
    for kategori, kode in zip(le.classes_, le.transform(le.classes_)):
        mapping_rows.append({'Fitur': col, 'Kategori Asli': kategori, 'Kode': kode})

df_mapping = pd.DataFrame(mapping_rows)

print("\n=== TABEL 2: Mapping Encoding Semua Fitur ===")
print(f"Jumlah baris tabel mapping: {len(df_mapping)}")
display(df_mapping)

=== TABEL 1: Data Hasil Encoding ===


,Age,Salt_Intake,Stress_Score,BP_History,Sleep_Duration,BMI,Medication,Family_History,Exercise_Level,Smoking_Status,Has_Hypertension
0,69,8.0,9,1,6.4,25.8,3,1,1,0,1
1,32,11.7,10,1,5.4,23.4,3,0,1,0,0
2,78,9.5,3,1,7.1,18.7,3,0,2,0,0
3,38,10.0,10,0,4.2,22.1,0,0,1,0,1
4,41,9.8,1,2,5.8,16.2,4,0,2,0,0



=== TABEL 2: Mapping Encoding Semua Fitur ===
Jumlah baris tabel mapping: 17


,Fitur,Kategori Asli,Kode
0,BP_History,Hypertension,0
1,BP_History,Normal,1
2,BP_History,Prehypertension,2
3,Medication,ACE Inhibitor,0
4,Medication,Beta Blocker,1
5,Medication,Diuretic,2
6,Medication,None,3
7,Medication,Other,4
8,Family_History,No,0
9,Family_History,Yes,1


## 3.4 Menyimpan Data Prepared

In [23]:
os.makedirs('../Datasets', exist_ok=True)
df_prepared.to_csv('../Datasets/prepared_dataset.csv', index=False)
print('Data berhasil disimpan ke ../Datasets/prepared_dataset.csv')

Data berhasil disimpan ke ../Datasets/prepared_dataset.csv


## 3.5 Contoh Train-Test Split
Sebagai ilustrasi awal, kita akan membagi dataset menjadi 3 skenario berbeda: 70-30, 80-20, dan 90-10. Nantinya, pada tahapan *Modeling*, kita akan membandingkan performa model pada ketiga split ini.

In [24]:
X = df_prepared.drop('Has_Hypertension', axis=1)
y = df_prepared['Has_Hypertension']

# Split 70-30
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Skenario 70-30 -> X_train shape:', X_train_70.shape, '| X_test shape:', X_test_70.shape)

# Split 80-20
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Skenario 80-20 -> X_train shape:', X_train_80.shape, '| X_test shape:', X_test_80.shape)

# Split 90-10
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
print('Skenario 90-10 -> X_train shape:', X_train_90.shape, '| X_test shape:', X_test_90.shape)

Skenario 70-30 -> X_train shape: (1389, 10) | X_test shape: (596, 10)
Skenario 80-20 -> X_train shape: (1588, 10) | X_test shape: (397, 10)
Skenario 90-10 -> X_train shape: (1786, 10) | X_test shape: (199, 10)
